# Coding with AI

This chapter is a gentle, assumes-nothing introduction to using **AI coding tools**
-- like Cursor, GitHub Copilot, Claude, or ChatGPT -- to build scientific software in
the brain and behavioral sciences.

You do **not** need to know anything about AI to start. We will define every term as
we go: what a "model" is, what a "prompt" is, what an "agent" does, what a "context
file" is, and so on. The goal is to make you a careful, effective user of these
tools -- someone who gets the productivity benefits *without* quietly introducing
errors into published science.

The chapter ends with an honest, two-sided discussion of when AI helps scientific
coding, when it hurts, and the methodological responsibilities that come with using
it. If you read only one section, read that one.

## 1. What is an AI coding assistant?

An AI coding assistant is a program that has read an enormous amount of text and
code and learned to predict plausible continuations. When you describe what you
want (or start typing code), it suggests code that *statistically tends to follow*
in similar situations. That is genuinely powerful -- and it is also the source of
every pitfall in this chapter. The tool is optimizing for *plausible-looking*, not
*correct*.

You will meet AI help in three forms:

* **Autocomplete (tab completion).** As you type, the tool suggests the rest of the
  line or block. You accept with Tab. Fast, low-stakes, easy to review.
* **Chat.** You ask a question in plain language ("how do I compute a PSTH?") and get
  an answer with code you can copy. Good for explanations and small snippets.
* **Agent.** You give a higher-level instruction ("add a function that loads the
  behavioral CSV and returns mean RT per condition") and the tool reads your files,
  writes code across them, runs commands, and iterates. The most powerful and the
  one that most needs supervision.

## 2. A vocabulary you actually need

* **Model.** The underlying AI (e.g. "Claude", "GPT"). Different models have
  different strengths, costs, and speeds. Bigger/newer is not automatically better
  for every task.
* **Prompt.** What you ask. The quality of the output depends enormously on the
  quality of the prompt: the more context and constraints you give, the better.
* **Context / context window.** The text the model can "see" at once -- your prompt,
  the open files, and recent conversation. It is finite. If your project is large,
  the model does not see all of it unless you point it at the relevant parts.
* **Context files / rules.** Many tools let you write a project file (e.g. a rules
  file or `AGENTS.md`) describing your conventions -- "use numpy, label all axes,
  always set a random seed." The assistant reads it and follows it. This is one of
  the highest-leverage things you can set up.
* **Tokens and usage.** Models process text in chunks called tokens, and most
  services bill (or rate-limit) by tokens used. Long files and long conversations
  cost more. Practically: keep requests focused.
* **Hallucination.** When a model confidently produces something false -- a function
  that does not exist, a citation that was never written, a formula that is subtly
  wrong. Hallucinations are not rare edge cases; they are a fundamental property of
  how these systems work, and detecting them is *your* job.

## 3. Getting set up (briefly)

This book is written with **Cursor** in mind -- an editor (a fork of VS Code) with AI
assistance built in -- but the principles apply to any tool.

1. Install the editor and open your project **folder** (not just a single file), so
   the assistant can see your data files and notebooks.
2. Try the three modes: accept a tab-completion, ask a question in chat, and give
   the agent a small, well-scoped task.
3. Create a short **rules/context file** for the project stating your conventions
   (next section shows what to put in it).

Many tools offer free tiers or student/educator access -- check current offerings, as
they change often. Nothing in this chapter depends on a paid plan.

## 4. How to work with AI well

A handful of habits separate "AI made me faster" from "AI introduced a bug I found
three months later."

* **Use version control from day one.** Put your project in `git`. Commit working
  states often. Then, when the AI changes ten files, you can *see exactly what
  changed* (`git diff`) and undo it if needed. This single habit makes AI safe to
  experiment with.
* **Keep raw data read-only.** Never let any script -- AI-written or not -- overwrite
  your raw data. Load it, copy it, transform the copy.
* **Work in small, reviewable steps.** Ask for one function, read it, test it, commit
  it. Giant "build the whole analysis" requests produce giant, unreviewable diffs.
* **Give the model real context.** Tell it the *shape and units* of your data
  ("`X` is trials x timepoints, RT in ms"), the *design* ("within-subject, unequal
  trials"), and the *goal*. Most silent errors come from the model guessing these.
* **Ask it to explain, not just produce.** "Explain what this line does and why" is
  often more valuable than the code itself, and it surfaces wrong assumptions.
* **Make it reproducible.** Insist on explicit random seeds, pinned package versions
  (`requirements.txt`), and code that runs top-to-bottom in a fresh kernel.
* **Read every line you keep.** If you would not be comfortable defending a line of
  code to your advisor, do not commit it. "The AI wrote it" is not a methods section.

:::{admonition} Modern Python
:class: tip
A good project rules file for a neuro/psych analysis might say: *use numpy and
pandas; always average within subject before across subjects; set a random seed and
print it; label all plot axes with units; never use exact `==` on floats; add a
short comment above each functional block.* The assistant will then apply these by
default -- turning your hard-won lessons into guardrails.
:::

## 5. The heart of the matter: code that runs but is wrong

This is the most important section in the chapter. AI assistants are very good at
producing code that *executes without error*. They are much less reliable at
producing code that is *scientifically correct*. The dangerous case is the overlap:
code that runs, returns a plausible number or a clean figure, and is **wrong**.

Below are real examples in the style you will actually encounter. In each, the
"AI-plausible" version runs fine. Your job -- the human-in-the-loop -- is to catch
why it is wrong.

### 5.1 The default t-test is not the one you want

In [1]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
# two groups with UNEQUAL variances and UNEQUAL sample sizes (very common)
group_a = rng.normal(500, 40, 30)
group_b = rng.normal(530, 120, 12)

# what an assistant typically writes:
t_default, p_default = stats.ttest_ind(group_a, group_b)
# what is usually correct for unequal variance (Welch's t-test):
t_welch, p_welch = stats.ttest_ind(group_a, group_b, equal_var=False)

print(f"default (Student, assumes equal variance): p = {p_default:.3f}")
print(f"Welch (does not assume equal variance):    p = {p_welch:.3f}")

default (Student, assumes equal variance): p = 0.024
Welch (does not assume equal variance):    p = 0.137


:::{admonition} Human check — know your function's defaults
:class: warning
`scipy.stats.ttest_ind` defaults to `equal_var=True` (Student's t-test), which
assumes the two groups have equal variances. With unequal variances and unequal n --
the norm in real data -- Welch's test (`equal_var=False`) is usually more
appropriate, and the p-values can differ enough to flip a conclusion. The default
code runs perfectly; only domain knowledge tells you to override the default.
:::

### 5.2 Many tests, no correction: significance by chance

In [2]:
rng = np.random.default_rng(1)
n_channels = 100
# PURE NOISE: there is no real difference in any channel
group_a = rng.normal(0, 1, (20, n_channels))
group_b = rng.normal(0, 1, (20, n_channels))

# loop a t-test over every channel (EEG electrodes, fMRI voxels, ...)
pvals = np.array([stats.ttest_ind(group_a[:, c], group_b[:, c]).pvalue
                  for c in range(n_channels)])

print("channels 'significant' at p < .05:", int((pvals < 0.05).sum()))
print("after Bonferroni correction:      ", int((pvals < 0.05 / n_channels).sum()))

channels 'significant' at p < .05: 3
after Bonferroni correction:       0


:::{admonition} Human check — multiple comparisons
:class: warning
Run 100 tests on pure noise at p < .05 and roughly 5 will be "significant" by
chance alone. The looping code is correct Python and produces real p-values -- but
reporting those hits without correcting for multiple comparisons (Bonferroni, FDR,
cluster-based permutation, ...) is a serious methodological error. An AI will write
the loop; deciding on the correction is a scientific responsibility it cannot
discharge for you.
:::

### 5.3 Data leakage: an accuracy that is too good to be true

In [3]:
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(2)
# 40 samples, 500 features, and labels that are PURE NOISE (no real signal)
X = rng.normal(0, 1, (40, 500))
y = rng.integers(0, 2, 40)

# WRONG: pick the features most correlated with y using ALL the data,
# THEN cross-validate on those hand-picked features
corrs = np.array([abs(np.corrcoef(X[:, j], y)[0, 1]) for j in range(X.shape[1])])
top = np.argsort(corrs)[-10:]
leaked = cross_val_score(LogisticRegression(max_iter=1000), X[:, top], y, cv=5).mean()

# RIGHT: do feature selection INSIDE the cross-validation, via a pipeline
honest_pipe = make_pipeline(SelectKBest(f_classif, k=10),
                            LogisticRegression(max_iter=1000))
honest = cross_val_score(honest_pipe, X, y, cv=5).mean()

print(f"leaked accuracy (selection outside CV): {leaked:.2f}")
print(f"honest accuracy (selection inside CV):  {honest:.2f}")

leaked accuracy (selection outside CV): 0.90
honest accuracy (selection inside CV):  0.60


:::{admonition} Human check — data leakage in machine learning
:class: warning
On data with *no real signal*, selecting features using the whole dataset before
cross-validation can yield "decoding accuracy" well above chance -- a completely
illusory result. The honest pipeline, which selects features within each fold,
sits near 50%, as it should. Both versions run without error and print a tidy
number. Leakage (selecting, scaling, or tuning using the test data) is one of the
most common and most damaging errors in neuroimaging/ML, and AI assistants
reproduce it readily because the leaky version is shorter and "looks fine."
:::

### 5.4 A single outlier can manufacture a correlation

In [4]:
rng = np.random.default_rng(3)
x = rng.normal(0, 1, 30)
y = rng.normal(0, 1, 30)        # x and y are INDEPENDENT

# add one extreme point (a sensor glitch, a mis-keyed value, an artifact)
x_out = np.append(x, 8.0)
y_out = np.append(y, 8.0)

print(f"r without the outlier: {np.corrcoef(x, y)[0, 1]:+.2f}")
print(f"r with one outlier:    {np.corrcoef(x_out, y_out)[0, 1]:+.2f}")

r without the outlier: +0.01
r with one outlier:    +0.65


:::{admonition} Human check — look at the data, not just the statistic
:class: warning
A single artifactual point can drive Pearson's r from ~0 to strongly positive. The
code is correct; the conclusion ("x and y are correlated!") is an artifact. No
function will warn you. Always plot the scatter, check for outliers, and consider
robust methods (e.g. Spearman) -- judgments a statistic alone cannot make.
:::

:::{admonition} Human check — and the quiet one: irreproducibility
:class: warning
Code that uses randomness (shuffles, splits, simulations, model initialization)
without a fixed seed gives a different answer every run. It never errors -- it just
cannot be reproduced, by you or by a reviewer. Always set and report an explicit
seed (`rng = np.random.default_rng(SEED)`), and pin your package versions.
:::

## 6. Prompts and review habits that reduce risk

You cannot prompt your way to guaranteed correctness, but good prompts and review
habits dramatically lower the error rate:

* **State the science in the prompt.** "Within-subject design, unequal trials per
  subject; average within subject first." "Variances are unequal." "Correct for
  multiple comparisons across 64 channels." If you do not say it, the model guesses.
* **Ask for the assumptions.** "List the statistical assumptions this code makes and
  where it could be wrong." This turns the model into a checklist generator.
* **Request a test or a sanity check.** "Add an assertion that the array is
  trials x time" or "show me the result on a tiny example where I know the answer."
* **Diff and review.** Read `git diff` for every AI change. If a change touches a file
  you did not expect, stop and look.
* **Verify against a known result.** Reproduce a number from a paper, or compute a
  toy case by hand, before trusting the pipeline on real data.
* **Prefer boring, explicit code.** If the AI writes something clever you do not
  fully understand, ask it to rewrite it simply. You have to maintain it.

## 7. A practical checklist

Before you trust AI-written analysis code, confirm:

- [ ] The project is in `git` and you have reviewed the `diff`.
- [ ] Raw data is untouched; transformations act on copies.
- [ ] Array axes and data shapes are what you think (`assert` them).
- [ ] Random seeds are set and reported; package versions are pinned.
- [ ] Statistical choices (test, error bars, corrections, exclusions) are *your*
      deliberate decisions, documented in comments and the methods section.
- [ ] You validated the pipeline against a known/hand-computed result.
- [ ] You can explain every line you kept.

## 8. Pros, cons, and methodological considerations

AI coding tools are neither a magic productivity button nor a threat to be banned.
They are a powerful instrument with a specific failure profile, and using them
responsibly in science means understanding both sides honestly.

### Where AI genuinely helps scientific coding

* **Lowering the barrier to entry.** A psychology or neuroscience student with a great
  question but limited programming background can get unstuck quickly, prototype an
  analysis, and learn by reading explanations. This is real democratization.
* **Speed on boilerplate.** Loading files, reshaping arrays, wiring up plots,
  writing argument parsers -- the tedious scaffolding that is easy to get right and
  slow to type. AI is excellent here.
* **Explanation and learning.** A patient tutor that explains an unfamiliar function,
  a cryptic error message, or someone else's code, available at 2 a.m.
* **Refactoring and documentation.** Turning a working-but-messy script into clear,
  commented, tested code; drafting docstrings; suggesting edge cases you missed.
* **Breadth.** Surfacing libraries and idioms you did not know existed (which you
  then verify).

### Where AI hurts, or can

* **Confident wrongness.** As Section 5 showed, the characteristic failure is code
  that runs and looks right but is scientifically incorrect. This is more dangerous
  than code that crashes, because nothing flags it.
* **Plausible fabrication.** Models invent function names, arguments, citations, and
  statistical justifications that do not exist. In a field built on accurate
  references and methods, this is corrosive.
* **Entrenching bad defaults and bias.** Models reproduce the most common patterns in
  their training data -- including common *mistakes* (the naive d', the uncorrected
  multiple comparisons, the leaky pipeline) and biases baked into that data.
* **Skill atrophy and shallow understanding.** If you never struggle through writing
  an analysis, you may not develop the judgment needed to recognize when the analysis
  is wrong -- exactly the judgment science depends on.
* **Homogenization.** If everyone accepts the same suggested approach, methodological
  monocultures (and their blind spots) can spread.

### Methodological considerations for AI-assisted science

These are the issues to think about *before* AI-generated code ends up in a thesis or
a paper:

* **Reproducibility and provenance.** Reproducibility is a property of the *code and
  data*, not of who or what wrote them -- but it requires discipline: seeds, pinned
  environments, and code that runs end-to-end. Keep a record of what you ran. Some
  groups note in their methods that AI assistance was used; norms are still forming,
  and you should follow your field's and institution's emerging guidance.
* **Validation is non-negotiable.** Treat AI-written analysis code the way you would
  treat a new instrument: calibrate it against known results before you trust its
  measurements. The burden of correctness is on you, the author.
* **Transparency in reporting.** Your methods section must describe what the analysis
  actually does -- the tests, corrections, exclusion criteria, and parameters -- in
  enough detail to reproduce, regardless of how the code was written. "We used a
  script the AI generated" is not a method.
* **Authorship, credit, and integrity.** AI tools are not authors and cannot be
  accountable for a paper's claims. Submitting AI-generated work as your own
  understanding (e.g. on an assignment) may violate academic-integrity policies;
  check your course's and institution's rules. Accountability for every result
  remains entirely human.
* **Licensing and plagiarism.** AI may reproduce code that is copyrighted or licensed
  in ways incompatible with your project, sometimes verbatim. Be cautious about
  pasting substantial generated blocks without understanding their provenance.
* **Data privacy, ethics, and the IRB.** Sending data or code to a cloud AI service
  can violate participant consent, data-use agreements, IRB protocols, HIPAA/FERPA,
  or GDPR -- especially with human-subjects data containing identifiers. Never paste
  participant data (or anything personally identifying) into an external AI tool
  unless you are certain it is permitted. When in doubt, ask your IRB or data
  steward, and prefer local/offline tools for sensitive data.
* **Equity of access.** The best tools cost money. Be mindful that AI assistance can
  widen gaps between well-resourced and under-resourced labs and students.
* **Environmental cost.** Large models consume substantial energy. It is reasonable
  to use them deliberately rather than reflexively.

### A balanced bottom line

Used well, AI coding tools can make you faster, help you learn, and remove drudgery.
Used uncritically, they will help you produce wrong results faster and more
confidently than ever before. The deciding factor is not the tool -- it is whether a
knowledgeable human stays genuinely in the loop: understanding the science, reading
the code, validating the results, and taking responsibility for what gets published.

That responsibility is also the reason this entire book is full of **Human check**
callouts. Learn to recognize the moments they mark, and you will be able to use AI as
what it is at its best: a fast, tireless, occasionally wrong assistant to a careful
scientist.

## Further reading

* Your institution's and field's current guidance on AI use in research and
  coursework (these are evolving quickly -- check for the latest).
* The documentation for any statistical function you use -- *read the defaults*.
* Reproducibility resources: `git` for version control, `requirements.txt`/
  environment files for pinned dependencies, and the reproducibility checklists many
  journals now publish.
* The full chapters of this book (P01-P09) and their "Applications & Modern Python"
  companions, which flag the specific code patterns most likely to be wrong.